# Matched run v2: runs in the background, survives a dropped browser
The generation runs as a background process on Colab's machine and writes to a log on Drive. If the page says "Connecting", the job keeps going. Check on it with the **status** cell, or open `matched_run.log` in Google Drive.
Secret needed: `GITHUB_TOKEN_DISTILL`. GPU: T4, L4 or A100 (any). When finished, use Runtime > Manage sessions > Terminate so units stop being used.

## 1. Setup (run each once after a reconnect)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, subprocess
TOKEN = userdata.get('GITHUB_TOKEN_DISTILL')
REPO = 'dedeepyakm/reasoning-trace-distillation'
if not os.path.exists('/content/repo'):
    subprocess.run(['git','clone',f'https://{TOKEN}@github.com/{REPO}.git','/content/repo'],check=True)
    subprocess.run(['git','-C','/content/repo','remote','set-url','origin',f'https://github.com/{REPO}.git'],check=True)
else:
    subprocess.run(['git','-C','/content/repo','pull',f'https://{TOKEN}@github.com/{REPO}.git','main'],check=True)
DRIVE_DATA = '/content/drive/MyDrive/reasoning-distillation/data'
os.makedirs(DRIVE_DATA, exist_ok=True)
if not os.path.exists('/content/repo/data'):
    os.symlink(DRIVE_DATA, '/content/repo/data')
os.chdir('/content/repo')
print(subprocess.run(['git','log','--oneline','-3'],capture_output=True,text=True).stdout)

In [ ]:
!pip -q install transformers accelerate datasets matplotlib bitsandbytes
!nvidia-smi

In [ ]:
import sys, json
sys.path.insert(0, '/content/repo')
from common.run_logging import run_step
PY = 'python'
OUT = 'data/traces_reasoning_matched_k5.jsonl'
LOG = '/content/drive/MyDrive/reasoning-distillation/matched_run.log'

## 2. Start the run in the background (safe to press twice: it will not start a second copy)

In [ ]:
running = subprocess.run('pgrep -f teacher_trace_generation_reasoning_matched', shell=True, capture_output=True, text=True).stdout.split()
if running:
    print('Already running (pid', running, '). Not starting another. Use the status cell.')
else:
    cmd = ['nohup', 'python', 'generation/teacher_trace_generation_reasoning_matched.py',
           '--problems-from', 'data/cut_instruct_final.jsonl', '--out', OUT,
           '--k', '5', '--max-new-tokens', '16384', '--precision', '4bit']
    proc = subprocess.Popen(cmd, stdout=open(LOG, 'a'), stderr=subprocess.STDOUT,
                            cwd='/content/repo', start_new_session=True)
    print('Started, pid', proc.pid, '\nLog on Drive:', LOG)

## 3. Status (run any time; also works right after a reconnect)

In [ ]:
n_running = len(subprocess.run('pgrep -f teacher_trace_generation_reasoning_matched', shell=True, capture_output=True, text=True).stdout.split())
print('process running:', 'YES' if n_running else 'NO')
print(subprocess.run('echo; echo "--- last log lines ---"; tail -n 12 ' + LOG + '; echo; echo "--- traces saved (target 250) ---"; wc -l < ' + OUT + '; echo; nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv', shell=True, capture_output=True, text=True).stdout)

## 4. After it finishes (process running: NO and 250 traces)
Truncation check, then cut analysis, problem-set check, and figures.

In [ ]:
rows = [json.loads(l) for l in open(OUT)]
n = len(rows)
hit = sum(r['hit_token_limit'] for r in rows)
noans = sum(r['predicted_answer'] is None for r in rows)
print(f'traces: {n}  problems: {len({r["id"] for r in rows})}')
print(f'hit token limit: {hit} ({100*hit/n:.1f}%)   no extractable answer: {noans} ({100*noans/n:.1f}%)')
print('max tokens used:', max(r['n_generated_tokens'] for r in rows))

In [ ]:
run_step('analyze_matched', [PY,'analysis/analyze_traces_v3.py','--trace-file',OUT,'--out','data/cut_reasoning_matched.jsonl','--with-difficulty'])

In [ ]:
run_step('compare_matched', [PY,'diagnostics/compare_problem_sets.py','--a','data/cut_instruct_final.jsonl','--b','data/cut_reasoning_matched.jsonl'])

In [ ]:
run_step('figures_matched', [PY,'reporting/make_figures.py','--cut-file','data/cut_reasoning_matched.jsonl','--label','reasoning_matched','--outdir','data/figures'])
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('data/figures/reasoning_matched_*.png')):
    print(p); display(Image(p))

## 5. Stop the run (only if you want to abort)

In [ ]:
subprocess.run('pkill -f teacher_trace_generation_reasoning_matched', shell=True)
print('stop signal sent')

## 6. Save the logs to GitHub (logs only)

In [ ]:
subprocess.run(['git','add','logs'])
print(subprocess.run(['git','status','--short'],capture_output=True,text=True).stdout)

In [ ]:
subprocess.run(['git','-c','user.name=Dedeepya Korukonda','-c','user.email=korukondadedeepya2002@gmail.com','commit','-m','Matched run logs'])
subprocess.run(['git','push',f'https://{TOKEN}@github.com/{REPO}.git','main'])